# 20 — Report readiness check

Fail loudly if the report bundle is missing core evidence, if only provisional seed trade exists, or if the price hypothesis has not been analysed.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

import json


In [ ]:
manifest_path = PATHS.report_inputs / "report_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Run notebook 19 first.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
from portugal_refining_resilience.readiness import build_readiness_checks

checks = build_readiness_checks(PATHS, manifest_path=manifest_path)
readiness = pd.DataFrame(checks)
persist_dataframe(readiness, PATHS.metrics / "report_readiness.csv", key_columns=["check"])
display(readiness)
publication_blockers = ["core_report_bundle_complete", "trade_not_seed_only", "jodi_annual_completeness_valid", "monthly_event_panel_valid", "monthly_event_outputs_valid", "eurostat_balance_available", "dgeg_trade_reconciliation_valid"]
if not readiness.loc[readiness["check"].isin(publication_blockers), "passed"].all():
    raise RuntimeError("Core publication-readiness checks failed. See data/metrics/report_readiness.csv")


A failed price co-movement check does not block a purely physical-supply report, but it **does** block any conclusion that refining reconfiguration changed domestic price exposure. DGEG reconciliation, monthly event models and JODI annual-completeness diagnostics are publication blockers for physical import-dependence claims.
